# `examples_usage` style scenario (single-move DRL step)

This notebook reproduces the spirit of `examples_usage` with:
- 2 relevant sender ports (like queue 1 and queue 2)
- 4 streams with periods inspired by the example
- a simple topology
- **single move per action** enabled in the env
- inline schedule plots using `Schedule.plot()` from `schedule.py`
- frame-by-frame snapshots + GIF

In [5]:
%matplotlib inline

import os
import copy
import tempfile

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display, Image as IPyImage

from usecase_generator import Usecase
from tsn_scheduling_problem import TSNScheduling
from env import TSNDefragEnv

In [6]:
def build_examples_usage_style_usecase(config_file: str = "uc_config.cfg"):
    """
    Topology with exactly two relevant egress ports:
      - host-1.Port1  -> host-2   (queue/port 1)
      - host-3.Port1  -> host-4   (queue/port 2)

    Streams 0,1 share port 1; streams 2,3 share port 2.
    This mirrors the structure in examples_usage.
    """
    G = nx.DiGraph()

    # Hosts with a single egress port each
    for h in ["host-1", "host-2", "host-3", "host-4"]:
        G.add_node(
            h,
            type="host",
            ports={"Port1": {"name": "Port1", "speed": 1000, "queueCount": 6}},
        )

    # Add bidirectional links so Usecase.ports can resolve every host.Port1.
    # Streams only use host-1.Port1 and host-3.Port1 as sender egress ports.
    G.add_edge(
        "host-1", "host-2",
        sourcePort="host-1.Port1", destPort="host-2.Port1",
        sourceNode="host-1", destNode="host-2",
        propDelay=10, linkBw=1000,
    )
    G.add_edge(
        "host-2", "host-1",
        sourcePort="host-2.Port1", destPort="host-1.Port1",
        sourceNode="host-2", destNode="host-1",
        propDelay=10, linkBw=1000,
    )
    G.add_edge(
        "host-3", "host-4",
        sourcePort="host-3.Port1", destPort="host-4.Port1",
        sourceNode="host-3", destNode="host-4",
        propDelay=10, linkBw=1000,
    )
    G.add_edge(
        "host-4", "host-3",
        sourcePort="host-4.Port1", destPort="host-3.Port1",
        sourceNode="host-4", destNode="host-3",
        propDelay=10, linkBw=1000,
    )

    # Periods inspired by examples_usage (scaled to ns)
    # 100, 200, 300, 600 -> here in "ns units" as 100_000 etc.
    periods = [100_000, 200_000, 300_000, 600_000]

    # Choose frame sizes so slot durations are visible.
    # duration_ns ~= 8.8 * maxFrameSize (at 1000 Mbps with tolerance 1.1)
    frame_sizes = [1200, 8500, 600, 1200]

    streams = [
        {
            "name": "stream-0",
            "type": "isochronous",
            "interval": periods[0],
            "maxFramesPerInterval": 1,
            "maxFrameSize": frame_sizes[0],
            "maxLatency": 1_000_000,
            "tlkrPssblPlcmnt": ["host-1"],
            "lstnsrPssblPlcmnt": [["host-2"]],
            "pathLists": [["host-1.Port1", "host-2"]],
            "earliest-transmit-offset": 0,
            "latest-transmit-offset": periods[3] // 2,
            "jitter": 0,
        },
        {
            "name": "stream-1",
            "type": "isochronous",
            "interval": periods[1],
            "maxFramesPerInterval": 1,
            "maxFrameSize": frame_sizes[1],
            "maxLatency": 1_000_000,
            "tlkrPssblPlcmnt": ["host-1"],
            "lstnsrPssblPlcmnt": [["host-2"]],
            "pathLists": [["host-1.Port1", "host-2"]],
            "earliest-transmit-offset": 0,
            "latest-transmit-offset": periods[3] // 2,
            "jitter": 0,
        },
        {
            "name": "stream-2",
            "type": "isochronous",
            "interval": periods[2],
            "maxFramesPerInterval": 1,
            "maxFrameSize": frame_sizes[2],
            "maxLatency": 1_000_000,
            "tlkrPssblPlcmnt": ["host-3"],
            "lstnsrPssblPlcmnt": [["host-4"]],
            "pathLists": [["host-3.Port1", "host-4"]],
            "earliest-transmit-offset": 0,
            "latest-transmit-offset": periods[3] // 2,
            "jitter": 0,
        },
        {
            "name": "stream-3",
            "type": "isochronous",
            "interval": periods[3],
            "maxFramesPerInterval": 1,
            "maxFrameSize": frame_sizes[3],
            "maxLatency": 1_000_000,
            "tlkrPssblPlcmnt": ["host-3"],
            "lstnsrPssblPlcmnt": [["host-4"]],
            "pathLists": [["host-3.Port1", "host-4"]],
            "earliest-transmit-offset": 0,
            "latest-transmit-offset": periods[3] // 2,
            "jitter": 0,
        },
    ]

    uc = Usecase(configFile=config_file, topology=G, streams=streams)
    uc.index = 0
    uc.config.set("streams_config", "numplacestalker", "1")
    uc.config.set("streams_config", "numplaceslistener", "1")
    uc.config.set("streams_config", "routing", "{'fixPath': True, 'shortestPath': True}")
    return uc

In [7]:
problem = TSNScheduling(problem_definition="uc_config.cfg")
usecase = build_examples_usage_style_usecase()

env = TSNDefragEnv(
    problem=problem,
    usecase=usecase,
    gap_threshold_ratio=0.05,
    move_penalty=0.01,
    fragmentation_metric="gap_count",
    single_move_per_step=True,  # <-- key requirement
)

obs, _ = env.reset()
print("Ports:", env.n_ports, "Streams:", env.n_streams, "Cycle:", env.cycle_ns)
print("Port id to name:")
for pid in range(1, env.n_ports + 1):
    print(pid, env._port_name[pid])

KeyError: 'host-2.Port1'

In [4]:
# Build initial offsets inspired by examples_usage:
# queue/port 1 -> streams 0 and 1 at [0, 10k]
# queue/port 2 -> streams 2 and 3 at [85k, 90k]
# (all in ns)
initial_offsets = [0.0, 10_000.0, 85_000.0, 90_000.0]
for s_idx, off in enumerate(initial_offsets):
    env._place_stream(s_idx, off)

print("Initial gap_count:", env._count_gaps())
sched0 = env.get_combined_schedule()
fig = sched0.plot(tick=1_000, label=True)
fig.suptitle("Initial schedule (examples_usage style)")
plt.show()

NameError: name 'env' is not defined

In [ ]:
def make_single_move_action(env, stream_idx, target_first_offset_ns):
    """
    Build one action matrix with exactly one non-zero entry.
    In single_move_per_step mode this guarantees one move or no move.
    """
    action = np.zeros((env.n_streams, env.n_ports), dtype=np.float32)
    first_pid = env.stream_info[stream_idx]["path"][0]
    pidx = first_pid - 1
    action[stream_idx, pidx] = target_first_offset_ns / env.cycle_ns + 1.0
    return action


# Quick proof: only one stream can move.
a = make_single_move_action(env, stream_idx=1, target_first_offset_ns=30_000.0)
_, r, _, _, info = env.step(a)
print("Reward:", r)
print("Moved stream:", info["moved_stream"], "n_moved:", info["n_moved"])

In [ ]:
# Reset to initial state for frame-by-frame demo
obs, _ = env.reset()
for s_idx, off in enumerate(initial_offsets):
    env._place_stream(s_idx, off)

# Deterministic action script: one stream move each frame
scripted_moves = [
    (1, 20_000.0),
    (3, 110_000.0),
    (0, 5_000.0),
    (2, 95_000.0),
    (1, 25_000.0),
]

tmp_dir = tempfile.mkdtemp(prefix="examples_usage_single_move_")
frame_paths = []
logs = []

# Frame 0 (before moves)
fig0 = env.get_combined_schedule().plot(tick=1_000, label=True)
fig0.suptitle("Frame 0: initial")
p0 = os.path.join(tmp_dir, "frame_00.png")
fig0.savefig(p0, dpi=110, bbox_inches="tight")
plt.close(fig0)
frame_paths.append(p0)
logs.append((0, None, None, env._count_gaps()))

for i, (s_idx, target_off) in enumerate(scripted_moves, start=1):
    action = make_single_move_action(env, s_idx, target_off)
    _, reward, _, _, info = env.step(action)

    fig = env.get_combined_schedule().plot(tick=1_000, label=True)
    fig.suptitle(
        f"Frame {i}: move s{s_idx} -> {int(target_off)} ns | reward={reward:.4f} | gaps={info['all_gaps']}"
    )
    p = os.path.join(tmp_dir, f"frame_{i:02d}.png")
    fig.savefig(p, dpi=110, bbox_inches="tight")
    plt.close(fig)
    frame_paths.append(p)

    logs.append((i, info["moved_stream"], reward, info["all_gaps"]))

logs

In [ ]:
display(IPyImage(filename=frame_paths[0]))
display(IPyImage(filename=frame_paths[-1]))

In [ ]:
# Create and display GIF inline
gif_path = os.path.join(tmp_dir, "single_move_episode.gif")
imgs = [PILImage.open(p).convert("P") for p in frame_paths]
imgs[0].save(
    gif_path,
    save_all=True,
    append_images=imgs[1:],
    duration=900,
    loop=0,
)
print("GIF path:", gif_path)
display(IPyImage(filename=gif_path))

In [ ]:
# Metrics trace
steps = [row[0] for row in logs]
rewards = [0.0 if row[2] is None else row[2] for row in logs]
gaps = [row[3] for row in logs]

fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax[0].plot(steps, rewards, marker="o")
ax[0].set_ylabel("Reward")
ax[0].grid(True, alpha=0.3)

ax[1].plot(steps, gaps, marker="o")
ax[1].set_ylabel("Gap count")
ax[1].set_xlabel("Frame")
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()